In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 Johnny Nuñez Cano
# SPDX-License-Identifier: MIT
# Relocated from the source revision recorded in ../MIGRATION_MANIFEST.json.
# See ../LICENSE and ../THIRD_PARTY_NOTICES.md.


# Move the two-cube box task to MuJoCo Warp


MuJoCo Warp runs MuJoCo physics on CUDA through Warp arrays and kernels. We keep the two-cube task from Part 1: the same robot profile, box, controller and physical acceptance checks.

First run the completed one-world reference for both robots. Then complete Steps 0–6 in `so101_mjwarp.py` to implement the CPU-to-GPU transfers and stepping loop yourself. The benchmark notebook separately measures the batched workflow.


## Select the GPU and reference

Use the Article 2 environment and start this notebook in `part2`. A CUDA device is required. Set `REFERENCE = False` once the exercise is complete.


## Open this lesson

Run from the checked-out tutorial in its Python 3.12 environment. In Colab, the setup fetches the selected public source and installs the hash-locked packages only in a fresh Python 3.12 kernel. Hosted execution has not been validated. Existing source files are preserved.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/johnnynunez/accelerated-computing-hub.git"
REF = "feature/blog2-validated-box-benchmark"
TUTORIAL_RELATIVE = Path("tutorials/sim2real-blogs/notebooks/mujoco")
COLAB_CHECKOUT = Path("/content/ach-mujoco-box-source")

if IN_COLAB:
    repo_root = COLAB_CHECKOUT
    git_env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
    if not repo_root.exists():
        repo_root.mkdir(parents=True)
        subprocess.run(["git", "init", str(repo_root)], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "remote", "add", "origin", REPO_URL], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "sparse-checkout", "init", "--cone"], check=True, env=git_env)
        subprocess.run(["git", "-C", str(repo_root), "sparse-checkout", "set", str(TUTORIAL_RELATIVE)], check=True, env=git_env)
    actual_remote = subprocess.check_output(["git", "-C", str(repo_root), "remote", "get-url", "origin"], text=True).strip()
    if actual_remote != REPO_URL:
        raise RuntimeError("Existing checkout uses another repository; choose a fresh COLAB_CHECKOUT.")
    subprocess.run(["git", "-C", str(repo_root), "fetch", "--depth", "1", "origin", REF], check=True, env=git_env)
    requested = subprocess.check_output(["git", "-C", str(repo_root), "rev-parse", "FETCH_HEAD"], text=True).strip()
    existing = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "--verify", "HEAD"], capture_output=True, text=True)
    if existing.returncode == 0 and existing.stdout.strip() != requested:
        raise RuntimeError("REF changed; existing files were preserved. Choose a fresh COLAB_CHECKOUT.")
    if existing.returncode != 0:
        subprocess.run(["git", "-C", str(repo_root), "checkout", "--detach", requested], check=True, env=git_env)
else:
    here = Path.cwd().resolve()
    repo_root = next((p for p in (here, *here.parents)
                      if (p / TUTORIAL_RELATIVE / "requirements.lock.txt").is_file()), None)
    if repo_root is None:
        raise RuntimeError("Open the notebook inside the accelerated-computing-hub checkout.")

tutorial_dir = repo_root / TUTORIAL_RELATIVE
lesson_dir = tutorial_dir / "part2"
if not (lesson_dir / "viewer_window.py").is_file():
    raise RuntimeError("This checkout does not contain the current box lessons.")
if IN_COLAB:
    if sys.version_info[:2] != (3, 12):
        raise RuntimeError("These teaching cells require a Python 3.12 kernel. Use the local/Brev environment; the benchmark notebook provides an isolated Python 3.12 runtime.")
    if any(name in sys.modules for name in ("mujoco", "mujoco_warp", "warp")):
        raise RuntimeError("Restart the runtime before installing the locked physics packages.")
    os.environ.setdefault("MUJOCO_GL", "egl")
    subprocess.run([sys.executable, "-m", "pip", "install", "--require-hashes", "-r", str(tutorial_dir / "requirements.lock.txt")], check=True)
os.chdir(lesson_dir)
if str(lesson_dir) not in sys.path:
    sys.path.insert(0, str(lesson_dir))
revision = subprocess.run(["git", "-C", str(repo_root), "rev-parse", "HEAD"], capture_output=True, text=True)
print("Source:", revision.stdout.strip() if revision.returncode == 0 else "Image snapshot without Git metadata; retain source fingerprints.")
print("Lesson directory:", lesson_dir)


In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import subprocess
import sys
import mujoco
import mujoco_warp as mjw
import warp as wp

PART2 = Path.cwd().resolve()
assert PART2.name == "part2" and (PART2 / "box_task.py").is_file()
REFERENCE = True
ROBOTS = ("so101", "rebot")
DEVICE = "cuda:0"  # Select one visible CUDA device.
wp.init()
assert wp.is_cuda_available() and wp.get_device(DEVICE).is_cuda, "This lesson requires a CUDA GPU."
SCRIPT = PART2 / ("solutions/so101_mjwarp_solution.py" if REFERENCE else "so101_mjwarp.py")
OUTPUT = PART2 / ".generated" / "box_lesson" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT.mkdir(parents=True)
print("Python:", sys.executable, "MuJoCo:", mujoco.__version__, "device:", DEVICE)


## Run the same box task on CUDA

Each run advances 2,000 control frames with 20 substeps per frame, covering 40 simulated seconds. The online controller and viewer path copy state between CPU and GPU so you can inspect correctness; these runs are not throughput measurements.

To watch the completed script from a terminal, omit `--headless-steps`. At the default 50 Hz, the box viewer closes after all 2,000 control frames and runs the physical task checks. Press Space to pause or resume, Esc or Q to cancel, or Ctrl+C in the terminal to stop. Cancelling before the full episode skips the final task check.


In [ ]:
for robot in ROBOTS:
    report = OUTPUT / f"{robot}_{DEVICE.replace(':', '')}.json"
    subprocess.run([sys.executable, str(SCRIPT), "--task", "box", "--robot", robot,
                    "--device", DEVICE, "--sim-substeps", "20", "--headless-steps", "2000",
                    "--report", str(report)], check=True)
    result = json.loads(report.read_text())
    assert result["success"], f"The physical box task failed: {report}"
    print(robot, "both cubes placed on", DEVICE)


## Implement the migration

Open `so101_mjwarp.py` and complete Steps 0–6. `box_task.py` supplies the shared scene, controller and observer. Model options and initialization are applied before `put_model`, so the device receives the intended task configuration.


### 0. Import MJWarp and Warp

At `TODO Step 0`, add the two imports. MJWarp is conventionally aliased to `mjw`:

```python
import mujoco_warp as mjw
import warp as wp
```

Note that `mujoco` is still imported above. MJWarp does not replace the standard bindings — you keep using them to compile MJCF and to run the CPU-side IK.

### 1. Upload the model to the GPU

At `TODO Step 1`, replace `m = None` with a call to `mjw.put_model(mjm)`.

This converts the compiled `mjModel` into an `mjw.Model` whose fields are Warp arrays in device memory. The model is read-only and shared across the whole batch, so this upload happens once regardless of how many worlds you simulate.

If the MJCF uses a feature MJWarp has not implemented, this call raises immediately rather than silently producing different physics. You can check what is supported in the [MuJoCo API compatibility table](https://github.com/google-deepmind/mujoco_warp#mujoco-api-compatibility).

### 2. Allocate batched device state

At `TODO Step 2`, replace `d = None` with a call to `mjw.make_data`.

This is where the GPU asks for something MuJoCo never did — explicit memory budgets, fixed up front:

| Parameter | Meaning |
|-----------|---------|
| `nworld` | Number of parallel worlds. Every `Data` field gains this as a leading dimension, so `d.qpos` has shape `(nworld, nq)`. |
| `nconmax` | Expected contacts **per world**. The global budget is `nworld * nconmax`, and one world may exceed its share as long as the total does not. |
| `njmax` | Maximum constraints **per world**. Unlike `nconmax`, this is a hard per-world limit. |

Use `nworld=1` here to inspect the same one-world task as Part 1, and pass the `nconmax` and `njmax` arguments the function already receives.

> These values are **not** parsed from the deprecated MJCF `size/nconmax` and `size/njmax` fields. You must pass them to `make_data` or `put_data`.

### 3. Seed the device state

At `TODO Step 3`, copy the host state onto the device and run a forward pass.

Two conversions happen on every copy:

- **Shape**: `mjd.qpos` has shape `(nq,)` but `d.qpos` has shape `(nworld, nq)`, so index with `[None, :]` to add the leading world axis.
- **Precision**: MJWarp is `float32`; MuJoCo defaults to double. Expect small numerical differences, and be careful with tight friction or convergence-sensitive setups.

Copy `qpos`, `qvel` and `ctrl`, then call `mjw.forward(m, d)` — the MJWarp equivalent of `mj_forward`, which recomputes derived quantities without advancing time.

### 4. Push the control vector to the device

At `TODO Step 4`, inside the substep loop, copy `mjd.ctrl` into `d.ctrl`.

In Part 1 this was a plain NumPy assignment into memory MuJoCo already owned. On the GPU there is no shared view, so the command must cross the bus. Use the same `[None, :]` reshape and `wp.copy`.

Doing this every substep is precisely the host-to-device traffic that destroys throughput. It is tolerable here only because parity mode runs one world and deliberately keeps the CPU in the loop.

### 5. Step the physics on the GPU

At `TODO Step 5`, call `mjw.step(m, d)`.

This one line replaces `mujoco.mj_step` and advances **every world in the batch** by one timestep. Batch size, contact work and hardware determine its cost; the next notebook measures that relationship.

### 6. Mirror world 0 back to the host

At `TODO Step 6`, copy `qpos` and `qvel` back so the CPU-side IK and the viewer stay in sync:

```python
mjd.qpos[:] = d.qpos.numpy()[0]
mjd.qvel[:] = d.qvel.numpy()[0]
```

`.numpy()` copies the Warp array back to host memory, and `[0]` selects world 0 out of the batch. Be aware that this is a synchronisation point: it blocks until the GPU finishes. In a real training loop you would keep observations on device and feed them straight to the policy.

## Read failures before timing

The physical report must pass both objects through grasp, lift, transport, release and settling. If contact or constraint capacity is exceeded, the run fails. Increase the relevant budget and rerun the whole task before trusting the result.


## Measure the batch comparison

Continue to `04__cpu_gpu_benchmark.ipynb`. It compares the complete box task at the same world counts on CPU and one selected GPU, with warm-up, repeated episodes and the same physical gates. One-world interactive latency and batched throughput answer different questions.


## Optional Step 7: capture the legacy stepping example

At `TODO Step 7`, in the `benchmark` function, record `mjw.step` as a CUDA graph.

`mjw.step` is dozens of small kernel launches, and each launch carries a fixed CPU cost. At small and medium world counts that overhead can dominate the actual physics. A CUDA graph records the whole launch sequence once and replays it as a single submission:

```python
graph = None
if device.is_cuda:
    with wp.ScopedCapture() as capture:
        mjw.step(m, d)
    graph = capture.graph
```

Three rules worth remembering:

- Capture **after** the state is initialised and after any `Model` field overrides for domain randomisation. The graph freezes the operations, not pointers you rebind later.
- Replays reuse the same buffers, so write new controls into the *same* arrays rather than reassigning `d.ctrl`.
- Graph capture is CUDA-only, hence the device check and the CPU fallback.
This small `benchmark` function retains the earlier stacking scene. Use the separate CPU/GPU benchmark notebook for the full two-cube box comparison.


In [ ]:
RUN_LEGACY_EXAMPLES = False
if RUN_LEGACY_EXAMPLES:
    for robot in ROBOTS:
        subprocess.run([sys.executable, str(SCRIPT), "--task", "stack", "--robot", robot,
                        "--device", DEVICE, "--headless-steps", "600"], check=True)
    subprocess.run([sys.executable, str(SCRIPT), "--benchmark", "--nworld", "256",
                    "--steps", "200", "--device", DEVICE], check=True)


## Next

Continue to Article 3 to construct the same task through Newton's model, state and control API. The box exercise uses `SolverMuJoCo`; the later cloth-and-cable exercise introduces coupled solvers for different materials.
